In [1]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from os.path import exists

sys.path.append('../..')

In [3]:
import numpy as np
from loguru import logger

from stable_baselines3 import PPO, DQN

In [23]:
from vimms.Common import POSITIVE, set_log_level_warning, load_obj, save_obj
from vimms.ChemicalSamplers import MZMLFormulaSampler, MZMLRTandIntensitySampler, MZMLChromatogramSampler
from vimms.Noise import UniformSpikeNoise
from vimms.Evaluation import evaluate_real
from vimms.Chemicals import ChemicalMixtureFromMZML
from vimms.Roi import RoiBuilderParams

from vimms.MassSpec import IndependentMassSpectrometer
from vimms.Controller import TopNController, TopN_SmartRoiController, WeightedDEWController
from vimms.Environment import Environment

from mass_spec_utils.data_import.mzmine import load_picked_boxes

from vimms_gym.env import DDAEnv
from vimms_gym.chemicals import generate_chemicals
from vimms_gym.evaluation import evaluate, run_method
from vimms_gym.common import METHOD_PPO, METHOD_DQN, METHOD_TOPN, METHOD_RANDOM, METHOD_FULLSCAN

# 1. Parameters

In [5]:
n_chemicals = (5000, 20000)
mz_range = (70, 1000)
rt_range = (0, 1440)
intensity_range = (1E4, 1E20)

In [6]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]
min_log_intensity = np.log(intensity_range[0])
max_log_intensity = np.log(intensity_range[1])

In [7]:
isolation_window = 0.7
N = 10
rt_tol = 120
small_rt_tol = 15
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE

enable_spike_noise = True
noise_density = 0.1
noise_max_val = 1e3

In [8]:
mzml_filename = '../fullscan_QCB.mzML'
samplers = None
samplers_pickle = 'samplers_fullscan_QCB.mzML.p'
if exists(samplers_pickle):
    logger.info('Loaded %s' % samplers_pickle)
    samplers = load_obj(samplers_pickle)
    mz_sampler = samplers['mz']
    ri_sampler = samplers['rt_intensity']
    cr_sampler = samplers['chromatogram']
else:
    logger.info('Creating samplers from %s' % mzml_filename)
    mz_sampler = MZMLFormulaSampler(mzml_filename, min_mz=min_mz, max_mz=max_mz)
    ri_sampler = MZMLRTandIntensitySampler(mzml_filename, min_rt=min_rt, max_rt=max_rt,
                                           min_log_intensity=min_log_intensity,
                                           max_log_intensity=max_log_intensity)
    roi_params = RoiBuilderParams(min_roi_length=3, at_least_one_point_above=1000)
    cr_sampler = MZMLChromatogramSampler(mzml_filename, roi_params=roi_params)
    samplers = {
        'mz': mz_sampler,
        'rt_intensity': ri_sampler,
        'chromatogram': cr_sampler
    }
    save_obj(samplers, samplers_pickle)

2022-05-02 12:59:57.509 | INFO     | __main__:<module>:5 - Loaded samplers_fullscan_QCB.mzML.p


In [9]:
params = {
    'chemical_creator': {
        'mz_range': mz_range,
        'rt_range': rt_range,
        'intensity_range': intensity_range,
        'n_chemicals': n_chemicals,
        'mz_sampler': mz_sampler,
        'ri_sampler': ri_sampler,
        'cr_sampler': cr_sampler,
    },
    'noise': {
        'enable_spike_noise': enable_spike_noise,
        'noise_density': noise_density,
        'noise_max_val': noise_max_val,
        'mz_range': mz_range
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
    }
}

In [10]:
max_peaks = 200
in_dir = 'results'

In [11]:
deterministic = True

# 2. Evaluation on QCB data

In [12]:
env_name = 'DDAEnv'

In [13]:
eval_dir = 'evaluation_QCB'
methods = [
    METHOD_PPO,
    METHOD_TOPN,
    METHOD_RANDOM,    
]
out_dir = eval_dir
in_dir, out_dir

('results', 'evaluation_QCB')

In [14]:
intensity_threshold = 0.5

#### Load pre-processed QCB chemicals

In [15]:
fullscan_file = '../fullscan_QCB.mzML'

In [16]:
# min_roi_intensity = 0
# min_roi_length = 0

# min_roi_intensity = 500
# min_roi_length = 0
# at_least_one_point_above = 5000

min_roi_intensity = 0
min_roi_length = 3
at_least_one_point_above = 1000

In [17]:
filename = 'datasets_%d_%d_%d.p' % (min_roi_intensity, min_roi_length, at_least_one_point_above)

if exists(filename):
    chemicals = load_obj(filename)
    print(len(chemicals))
else:
    rp = RoiBuilderParams(min_roi_intensity=min_roi_intensity, min_roi_length=min_roi_length, 
                   at_least_one_point_above=at_least_one_point_above)
    cm = ChemicalMixtureFromMZML(fullscan_file, roi_params=rp)
    chemicals = cm.sample(None, 2, source_polarity=ionisation_mode)
    print(len(chemicals))
    save_obj(chemicals, filename)

43107


#### Filter chemicals by mz and RT range

In [18]:
filtered = []
for chem in chemicals:
    if (min_mz < chem.isotopes[0][0] < max_mz) and (min_rt < chem.rt < max_rt):
        filtered.append(chem)
        
len(filtered)

43060

In [19]:
filtered_chem_list = [filtered]

#### Disable spike noise in DDAEnv

In [20]:
copy_params = dict(params)
copy_params['noise']['enable_spike_noise'] = False
copy_params

{'chemical_creator': {'mz_range': (70, 1000),
  'rt_range': (0, 1440),
  'intensity_range': (10000.0, 1e+20),
  'n_chemicals': (5000, 20000),
  'mz_sampler': <vimms.ChemicalSamplers.MZMLFormulaSampler at 0x7fc26a09b6d0>,
  'ri_sampler': <vimms.ChemicalSamplers.MZMLRTandIntensitySampler at 0x7fc27aec86a0>,
  'cr_sampler': <vimms.ChemicalSamplers.MZMLChromatogramSampler at 0x7fc27af25d60>},
 'noise': {'enable_spike_noise': False,
  'noise_density': 0.1,
  'noise_max_val': 1000.0,
  'mz_range': (70, 1000)},
 'env': {'ionisation_mode': 'Positive',
  'rt_range': (0, 1440),
  'isolation_window': 0.7,
  'mz_tol': 10,
  'rt_tol': 120}}

#### Test different methods

In [21]:
copy_params = dict(params)
copy_params['env']['rt_tol'] = rt_tol

for method in methods:
    banner = 'method = %s max_peaks = %d rt_tol = %d' % (method, max_peaks, rt_tol)
    print(banner)
    print()
    
    copy_params = dict(params)
    if method == 'PPO':
        fname = os.path.join(in_dir, '%s_%s.zip' % (env_name, method))
        model = PPO.load(fname)
        copy_params['env']['rt_tol'] = rt_tol
    elif method == 'DQN':
        fname = os.path.join(in_dir, '%s_%s.zip' % (env_name, method))
        model = DQN.load(fname)
        copy_params['env']['rt_tol'] = rt_tol        
    else:
        model = None
        copy_params['env']['rt_tol'] = small_rt_tol        

    env = DDAEnv(max_peaks, copy_params)
    run_method(env_name, copy_params, max_peaks, filtered_chem_list, method, out_dir, mzml_prefix='QCB',
               N=10, min_ms1_intensity=min_ms1_intensity, model=model, print_reward=True, print_eval=True)
    print()

method = PPO max_peaks = 200 rt_tol = 120


Episode 0 (43060 chemicals)
steps	 500 	total rewards	 75.81979129173445
steps	 1000 	total rewards	 204.58642550152246
steps	 1500 	total rewards	 339.14512933881775
steps	 2000 	total rewards	 486.13257274007185
steps	 2500 	total rewards	 651.9714582461861
steps	 3000 	total rewards	 793.3345136232737
steps	 3500 	total rewards	 875.0477267022175
steps	 4000 	total rewards	 938.8667098506643
steps	 4500 	total rewards	 983.8409490476392
steps	 5000 	total rewards	 1024.7163873914724
Finished after 5229 timesteps with total reward 1048.73196082227
{'coverage_prop': '0.058', 'intensity_prop': '0.037', 'ms1/ms2 ratio': '0.605', 'efficiency': '0.773', 'TP': '1108', 'FP': '391', 'FN': '41561', 'precision': '0.739', 'recall': '0.026', 'f1': '0.050'}

method = topN max_peaks = 200 rt_tol = 120


Episode 0 (43060 chemicals)
steps	 500 	total rewards	 27.30326929199802
steps	 1000 	total rewards	 140.17828673156436
steps	 1500 	total rewards	 296.3

#### Test classic Top-N controller in ViMMS

In [24]:
mass_spec = IndependentMassSpectrometer(ionisation_mode, filtered)

In [25]:
controller = TopNController(ionisation_mode, N, isolation_window, mz_tol, rt_tol, min_ms1_intensity)
env = Environment(mass_spec, controller, min_rt, max_rt, progress_bar=True, out_dir=out_dir, 
                  out_file='QCB_TopN_controller.mzML')
env.run()

  0%|          | 0/1440 [00:00<?, ?it/s]

#### Evaluation from mzML

Evaluation against the list of peaks picked from the fullscan QCB files.

There are two sets of parameters used for the peak picking: 'before' and 'after'.
'After' is more strict than 'before', with higher thresholds on intensity etc.

In [26]:
fullscan_path = os.path.abspath('../fullscan_QCB.mzML')
fullscan_path

'/Users/joewandy/Work/git/vimms-gym/notebooks/fullscan_QCB.mzML'

### 'Before' results

In [27]:
aligned_file = os.path.abspath('../fullscan_QCB_box_before.csv')
aligned_file

'/Users/joewandy/Work/git/vimms-gym/notebooks/fullscan_QCB_box_before.csv'

In [28]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_PPO_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.24248171227309673], [0.16325221705798484])

In [29]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.27363858033053373], [0.20956085121182907])

In [30]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_controller.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.4191276076943918], [0.2742782875513426])

In [31]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_random_0.mzML'),  
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.13058791655377947], [0.26411909882587625])

### 'After' results

In [32]:
aligned_file = os.path.abspath('../fullscan_QCB_box_after.csv')
aligned_file

'/Users/joewandy/Work/git/vimms-gym/notebooks/fullscan_QCB_box_after.csv'

In [33]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_PPO_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.5914786967418546], [0.33809991611541956])

In [34]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.5852130325814536], [0.4294251485704215])

In [35]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_controller.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.7969924812030075], [0.34534735857506416])

In [36]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_random_0.mzML'),  
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.5213032581453634], [0.47879808010150304])